# Basic Coalescent Model

Formålet med denne notebook er at implementere den klassiske Kingman coalescent som en kontinuert-tids Markov proces ved hjælp af phasic.

Coalescent-processen beskriver hvordan genetiske linjer samles bagud i tid, indtil alle individer deler en fælles stamfader (MRCA).

Jeg vil:

1. Definere state space
2. Definere transitions (coalescence events)
3. Konstruere modellen som en Markov jump process
4. Analysere dens egenskaber (moments, distribution)
5. Simulere data

In [2]:
import phasic

from phasic import Graph # ALWAYS import phasic first to set jax backend correctly
import numpy as np
import seaborn as sns
%config InlineBackend.figure_format = 'svg'
from vscodenb import set_vscode_theme

np.random.seed(42)
set_vscode_theme()
sns.set_palette('tab10')

## Kingman Coalescent

Vi betragter en sample på $n$ individer.

State:
State vektoren er:
$$
state=(x_1, x_2,\dots, x_n)
$$

hvor: $x_k =$ antal linjer med $k$ efterkommere

Initial state:
$$
(n, 0, ..., 0)
$$

Transition:
To linjer med størrelser $i$ og $j$ coalescerer til en linje med størrelse $i+j$.

Rate:
For $i ≠ j$:
$$
rate = x_i * x_j
$$

For $i = j$:
$$
rate = x_i * (\frac{x_i-1}{2})
$$

Dette kan samles som:
$$
rate = x_i * (\frac{x_j-\delta_{ij}}{1+\delta_{ij}})
$$

hvor $\delta_{ij}$ er Kronecker delta.

Processen stopper når:
$$
(0,0, \dots,1) (MRCA)
$$


Phasic konstruerer automatisk hele state space ved at udforske alle mulige transitions.

Grafen repræsenterer en kontinuert-tids Markov kæde, hvor: 

* noder = states
* edges = transitions
* vægte = transition rates

In [ ]:
from phasic import with_ipv

nr_samples = 4

@with_ipv([nr_samples]+[0]*(nr_samples-1))
def coalescent(state):
    transitions = []

    #iterer over alle par af lineage-typer
    for i in range(state.size):
        for j in range(i, state.size):
            same = int(i == j)

            #Check om coalescene er mulig   
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue

            #Ny state efter coalescene 
            new_state = state.copy()
            new_state[i] -= 1
            new_state[j] -= 1
            new_state[i+j+1] += 1

            #Coalescent rate
            rate = state[i]*(state[j]-same)/(1+same)

            transitions.append((new_state, rate))
    return transitions

# Konstruktion af graf
graph = Graph(coalescent)
graph.plot()

## State space analyse

In [ ]:
print("Antal states:", graph.vertices_length())
print("States:")
print(graph.states())

State space vokser hurtigt med $n$ og er den primære kilde til computational kompleksitet.

## Moments

In [ ]:
print("Expectation:", graph.expectation())
print("Variance:", graph.variance())
print("Moments:", graph.moments(5))

Phasic bruger phase-type distributioner til at beregne eksakte moments.

Dette er en stor fordel frem for simulation:

* ingen Monte Carlo fejl
* deterministiske resultater

## Fordelingsfunktioner

In [ ]:
t = np.linspace(0, 5, 200)

pdf = graph.pdf(t)
cdf = graph.cdf(t)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(t, pdf)
plt.title("PDF")

plt.subplot(1,2,2)
plt.plot(t, cdf)
plt.title("CDF")

plt.show()

PDF beskriver sandsynligheden for coalescent tid.
CDF beskriver sandsynligheden for at MRCA er nået før tid t.

## Simulation

In [ ]:
samples = graph.sample(5000)

print("Sample mean:", np.mean(samples))
print("Theoretical mean:", graph.expectation())

In [ ]:
plt.hist(samples, bins=50, density=True)
plt.title("Simulated coalescent times")
plt.xlabel("Time")
plt.ylabel("Density")
plt.show()

Simulation bruges senere til:

* inference
* model validation

## CPU monitoring

In [ ]:
%%monitor
graph.moments(10)

## Caching

In [ ]:
graph_cached = Graph(coalescent, cache=True)

Graph caching gemmer state space på disk og undgår genberegning.

Dette er essentielt for større modeller.

## Konklusion

* Coalescent kan formuleres som en Markov jump process
* States repræsenterer lineage-struktur
* Transition rates følger klassisk coalescent teori
* Phasic gør det muligt at:
  * beregne eksakte fordelinger
  * simulere data
  * lave inference

Dette bliver fundamentet for:

* migration modeller
* IM modeller
* ghost populations